# Cost Optimization for LLM Workloads

LLM API costs scale linearly with token consumption. At prototype scale this is invisible; at production scale — tens of thousands of queries per day — it becomes a budget line item that engineering must actively manage. This notebook develops a layered cost reduction stack: prompt caching to avoid re-sending identical prefixes, semantic caching to avoid re-querying the model for near-duplicate questions, intelligent model routing to reserve expensive models only for requests that genuinely need them, and the OpenAI Batch API for workloads that tolerate a 24-hour turnaround in exchange for a 50% discount.

The financial services context makes cost optimization non-trivial: compliance documents are sensitive (ruling out third-party semantic caching services), queries are heterogeneous (ruling out a one-model-fits-all policy), and some analyses are time-critical (ruling out batching for those). We build each layer as a composable class so they can be mixed and matched to fit a given workload's constraints.

Setup:

In [ ]:
#| echo: false
import os, json, time, hashlib
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Prompt Caching

**How OpenAI prompt caching works.** When the same prompt prefix of at least 1,024 tokens is sent within a short window, OpenAI's infrastructure can reuse the key-value (KV) cache from the previous request rather than recomputing attention over the prefix from scratch. Cached input tokens are billed at half the standard rate — a meaningful saving when the system prompt contains a long document, a large few-shot block, or a multi-page regulatory excerpt that stays constant across many user queries.

<br>

**Design implications.** To maximise cache hit rate: (1) place the **static** content (system instructions, large documents, few-shot examples) at the *beginning* of the prompt and the **dynamic** content (the user's actual query) at the *end* — OpenAI caches from the left; (2) keep the static prefix byte-for-byte identical across requests (even a single changed character busts the cache); (3) send requests within a few minutes to stay within the cache window. A common pattern in RAG is to embed the retrieved chunks *after* a large, fixed system prompt rather than before it, specifically to benefit from caching on the system prompt.

<br>

**Measuring cache hits.** The `usage` field in the API response exposes `prompt_tokens_details.cached_tokens`. We can log this alongside the normal token counts to track cache efficiency over time.

A helper that reports whether the OpenAI response was served from the prompt cache:

In [ ]:
import openai, os, time

client = openai.OpenAI()

# A large, static system prompt (≥1024 tokens in practice).
# We simulate this with a repeated regulatory excerpt.
LARGE_SYSTEM = (
    "You are a compliance analyst. Below are the full Basel III capital adequacy "
    "requirements (Articles 1-127). " + ("[regulatory text placeholder] " * 80) +
    "Given the filing excerpt provided by the user, identify any capital ratio "
    "violations and summarise the key risk disclosures."
)

def analyse_with_cache_stats(user_text: str) -> dict:
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": LARGE_SYSTEM},  # <1>
            {"role": "user",   "content": user_text},
        ],
        max_tokens=200,
        temperature=0.0,
    )
    elapsed = time.perf_counter() - t0
    usage   = resp.usage
    cached  = getattr(getattr(usage, "prompt_tokens_details", None), "cached_tokens", 0)  # <2>
    return {
        "cached_tokens":  cached,
        "prompt_tokens":  usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "latency_s":      round(elapsed, 3),
        "cache_hit_rate": round(cached / usage.prompt_tokens, 3) if usage.prompt_tokens else 0,
    }

1. The large static system prompt is placed first so OpenAI can cache the longest possible prefix. Only the `user` message changes between requests.
2. `prompt_tokens_details` is an optional nested object; we use `getattr` with a default of `0` to stay compatible with older response shapes.

Sending the same user query twice — the second call should show `cached_tokens > 0`:

In [ ]:
QUERY = "Our CET1 ratio is 12.1%. Is this compliant with Basel III minimums?"

for i in range(2):
    stats = analyse_with_cache_stats(QUERY)
    print(f"call {i+1}: {stats}")

## Semantic Cache

**What prompt caching does not solve.** OpenAI's prompt caching only helps when the *exact same byte sequence* is re-sent. It does nothing for semantically similar but lexically different queries — "What is our CET1 ratio?" and "How does our Common Equity Tier 1 compare to the minimum?" will both incur full token costs even though the answer is identical. Semantic caching addresses this by storing (query embedding, response) pairs and retrieving cached responses when a new query is sufficiently close in embedding space.

<br>

**Privacy constraint.** In financial services, a third-party semantic cache (e.g. GPTCache cloud) may not be acceptable since query text leaves the organisation's boundary. We implement a simple in-process cache backed by numpy — no external service. In production this can be backed by a Redis instance inside the security perimeter.

<br>

**Similarity threshold.** The cache lookup uses cosine similarity. A threshold of $0.95$ is conservative (only near-exact paraphrases hit the cache). Lowering the threshold to $0.90$ increases the hit rate but risks returning stale answers for queries that differ in financially material ways. Calibrating this threshold on a sample of query pairs is essential before raising it in production.

Implementing the `SemanticCache`:

In [ ]:
import numpy as np
from dataclasses import dataclass, field

@dataclass
class CacheEntry:
    query:     str
    embedding: np.ndarray
    response:  str

class SemanticCache:
    def __init__(self, threshold: float = 0.95):
        self.threshold = threshold
        self._store: list[CacheEntry] = []
        self._client = openai.OpenAI()
        self.hits = 0; self.misses = 0

    def _embed(self, text: str) -> np.ndarray:
        resp = self._client.embeddings.create(
            model="text-embedding-3-small", input=text)
        v = np.array(resp.data[0].embedding, dtype=np.float32)
        return v / np.linalg.norm(v)  # <1>

    def _cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b))    # <2>

    def get(self, query: str) -> str | None:
        if not self._store:
            self.misses += 1; return None
        qv    = self._embed(query)
        sims  = [self._cosine(qv, e.embedding) for e in self._store]
        best  = max(range(len(sims)), key=lambda i: sims[i])
        if sims[best] >= self.threshold:
            self.hits += 1
            return self._store[best].response
        self.misses += 1; return None

    def put(self, query: str, response: str):
        self._store.append(CacheEntry(
            query=query,
            embedding=self._embed(query),
            response=response,
        ))

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total else 0.0

1. Pre-normalising the embedding vector makes the dot product equivalent to cosine similarity — avoids the division in every cache lookup.
2. Both vectors are pre-normalised, so `dot(a, b) == cosine_similarity(a, b)` with no extra computation.

Testing the cache with semantically similar and dissimilar queries:

In [ ]:
cache = SemanticCache(threshold=0.92)

# Seed the cache with one query and a synthetic response
seed_q   = "What is our CET1 capital ratio and is it compliant?"
seed_ans = "Our CET1 ratio is 14.8%, well above the 4.5% Basel III minimum."
cache.put(seed_q, seed_ans)

test_queries = [
    "How does our Common Equity Tier 1 ratio compare to regulatory requirements?",  # near-duplicate
    "What are the primary operational risk factors disclosed in this filing?",       # dissimilar
    "Is our CET1 ratio above the minimum threshold?",                               # near-duplicate
]

for q in test_queries:
    hit = cache.get(q)
    status = "HIT " if hit else "MISS"
    print(f"{status} | {q[:60]}")

print(f"\ncache hit rate: {cache.hit_rate:.0%}")

## Model Router

**The cost-quality trade-off.** GPT-4o-mini costs roughly 17× less per token than GPT-4o but has measurably lower capability on complex reasoning tasks. The optimal strategy is not to use one model universally but to route each request to the cheapest model that is *sufficient* for it. Simple factual lookups and short summaries are well within GPT-4o-mini's capability; multi-step compliance analysis with conflicting precedents warrants GPT-4o. A `ModelRouter` operationalises this heuristic.

<br>

**Routing signals.** We use two signals: (1) **query complexity** — estimated by a fast classifier that looks at token count, the presence of financial keywords that indicate multi-step reasoning ("compare", "reconcile", "derive", "calculate"), and question structure; (2) **risk tier** — some query categories (regulatory capital adequacy, counterparty exposure) are designated high-risk and always routed to the strong model regardless of complexity estimate. This two-signal approach prevents the router from downgrading a genuinely high-stakes query just because it happens to be phrased simply.

<br>

**Measuring router accuracy.** The router should be validated on a labelled dataset where human experts have annotated which queries *require* GPT-4o to answer correctly. A router that over-routes to GPT-4o-mini on complex queries will surface as quality regressions; one that over-routes to GPT-4o will surface as unnecessary cost. Both failure modes are auditable.

Implementing the `ModelRouter`:

In [ ]:
import re
from enum import Enum

class Tier(str, Enum):
    FAST   = "gpt-4o-mini"
    STRONG = "gpt-4o"

# Keywords that signal multi-step or high-stakes reasoning
COMPLEX_SIGNALS = [
    r"\bcompare\b", r"\breconcile\b", r"\bderive\b", r"\bcalculate\b",
    r"\bexplain why\b", r"\bcontradict\b", r"\bconflict\b", r"\bstress.test\b",
]
HIGH_RISK_TOPICS = [
    "capital adequacy", "cet1", "leverage ratio", "counterparty exposure",
    "margin call", "regulatory capital", "stress scenario",
]

class ModelRouter:
    def __init__(self, token_threshold: int = 200):
        self.token_threshold = token_threshold  # rough word-count proxy
        self._complex_re   = re.compile("|".join(COMPLEX_SIGNALS), re.I)
        self._risk_re      = re.compile(
            "|".join(re.escape(t) for t in HIGH_RISK_TOPICS), re.I)

    def route(self, query: str, context: str = "") -> Tier:  # <1>
        full_text  = f"{query} {context}"
        word_count = len(full_text.split())

        if self._risk_re.search(full_text):          # <2>
            return Tier.STRONG

        complex_hits = len(self._complex_re.findall(query))
        if complex_hits >= 2 or word_count > self.token_threshold:
            return Tier.STRONG

        return Tier.FAST

    def select_client(self, query: str, context: str = "") -> LLMClient:
        tier = self.route(query, context)
        return LLMClient(model=tier.value)

1. `context` is included in the routing decision because the retrieved RAG chunks may contain high-risk keywords even if the user's question is phrased simply.
2. High-risk topic detection is checked first — it overrides all other signals to prevent downgrading compliance-critical queries.

Testing the router on a sample of financial queries:

In [ ]:
router = ModelRouter()

test_cases = [
    ("Summarise the revenue figure from the filing.",                              "FAST"),
    ("What is the CET1 ratio reported?",                                           "STRONG"),  # high-risk topic
    ("Compare and reconcile the two reported leverage ratios.",                    "STRONG"),  # complex signals
    ("How many employees does the firm have?",                                     "FAST"),
    ("Derive the net interest margin from the income statement and explain why "
     "it contradicts the prior year guidance.",                                    "STRONG"),  # multiple signals
]

print(f"{'Query':<65} {'Expected':<8} {'Got'}")
print("-" * 90)
for q, expected in test_cases:
    got    = router.route(q).name
    marker = "✓" if got == expected else "✗"
    print(f"{q[:64]:<65} {expected:<8} {got} {marker}")

## Batch API

**The asynchronous workload pattern.** Not every LLM call in a financial services system is user-facing and latency-sensitive. Nightly batch jobs that classify thousands of SEC filings, weekly risk report generation, or offline training data labelling are all batch workloads where a 24-hour turnaround is entirely acceptable. OpenAI's Batch API charges 50% of the standard rate for these workloads, making it the highest-leverage cost reduction available for offline pipelines.

<br>

**How it works.** We write a JSONL file where each line is a self-contained request object with a `custom_id` (for matching responses back to inputs), a `method` (`POST`), a `url` (the endpoint, e.g. `/v1/chat/completions`), and a `body` (the normal request payload). We upload the file, create a batch job, poll for completion, and download the results JSONL. Each result line carries the original `custom_id` alongside the API response.

<br>

**Error handling.** Some requests in a batch may fail individually (malformed prompt, content policy) without failing the whole batch. We always check `response.error` on each result line and log failures separately. A downstream reconciliation step matches the `custom_id` back to the original filing identifier so failed items can be retried or escalated.

Building and submitting a batch of five filing classification requests:

In [ ]:
import json, tempfile, pathlib

FILINGS = [
    ("AAPL-2024", "Net revenues were $391 billion. Return on equity was 160%."),
    ("GS-2024",   "Investment banking revenues decreased 23% to $6.1 billion."),
    ("JPM-2024",  "Our CET1 ratio was 15.3%, above the 4.5% regulatory minimum."),
    ("BAC-2024",  "Operational risk losses totalled $1.2 billion in the period."),
    ("MS-2024",   "We maintain an LCR of 134%, exceeding the 100% requirement."),
]

SYS = "Classify the following SEC filing excerpt as: RISK_HIGH, RISK_MEDIUM, or RISK_LOW."

def build_batch_file(filings: list[tuple[str, str]]) -> pathlib.Path:  # <1>
    lines = []
    for fid, text in filings:
        lines.append(json.dumps({
            "custom_id": fid,
            "method":    "POST",
            "url":       "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {"role": "system", "content": SYS},
                    {"role": "user",   "content": text},
                ],
                "max_tokens": 10,
                "temperature": 0.0,
            },
        }))
    path = pathlib.Path(tempfile.mktemp(suffix=".jsonl"))
    path.write_text("\n".join(lines))
    return path

batch_path = build_batch_file(FILINGS)
print(f"batch file: {batch_path} ({batch_path.stat().st_size} bytes)")
print("first line:", batch_path.read_text().splitlines()[0][:120])

1. Each line in the JSONL is an independent, self-contained API request. The `custom_id` is the only link between the input and the result — choose a value that maps back to your database primary key.

Uploading and submitting the batch, then polling until completion:

In [ ]:
import time

def submit_and_poll(batch_path: pathlib.Path, poll_interval: int = 30) -> list[dict]:
    batch_client = openai.OpenAI()

    # Upload the JSONL file
    with open(batch_path, "rb") as f:
        uploaded = batch_client.files.create(file=f, purpose="batch")  # <1>
    print(f"uploaded: {uploaded.id}")

    # Create the batch job
    batch = batch_client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    print(f"batch id: {batch.id}  status: {batch.status}")

    # Poll until complete
    while batch.status not in ("completed", "failed", "expired", "cancelled"):
        time.sleep(poll_interval)
        batch = batch_client.batches.retrieve(batch.id)
        print(f"  status: {batch.status}  "
              f"completed: {batch.request_counts.completed}/{batch.request_counts.total}")

    if batch.status != "completed":
        raise RuntimeError(f"Batch ended with status: {batch.status}")

    # Download and parse results
    content = batch_client.files.content(batch.output_file_id).text  # <2>
    results = [json.loads(line) for line in content.strip().splitlines()]
    return results

# Uncomment to run (will block for up to 24h, typically minutes for small batches):
# results = submit_and_poll(batch_path)
# for r in results:
#     cid  = r["custom_id"]
#     if r.get("error"):
#         print(f"{cid}: ERROR — {r['error']}")
#     else:
#         ans = r["response"]["body"]["choices"][0]["message"]["content"].strip()
#         print(f"{cid}: {ans}")

1. `purpose="batch"` is required; it tells OpenAI the file is a batch input rather than a fine-tuning dataset or assistant file.
2. `files.content(...).text` streams the result JSONL as a string. For large batches this may be several MB — consider streaming line-by-line using `.iter_lines()` in production.

:::{.callout-note}
The Batch API has a per-request timeout of 24 hours and a maximum of 50,000 requests or 200 MB per batch. For larger offline pipelines, split inputs into multiple batch jobs and fan-out the polling loop.

:::

## Cost-Quality Trade-off

**Putting it together.** The four techniques — prompt caching, semantic caching, model routing, and batch API — are complementary layers. A production system may apply all four simultaneously: the semantic cache is checked first (free); if missed, the router selects the model; the request is sent to the API (prompt caching happens automatically); time-insensitive requests are queued for the Batch API. The combined effect is a cost reduction of roughly 60–80% at scale with no degradation on the critical path.

<br>

**Measuring the trade-off empirically.** We simulate the expected cost and accuracy at each layer combination on a representative query set. Cost is computed from token usage; accuracy is estimated from the router's misrouting rate (false FAST routes that would have required STRONG). This gives a Pareto curve: cheapest configuration at each quality level.

Simulating the cost breakdown across strategy configurations:

In [ ]:
#| code-fold: true
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['svg']

# Simulated per-1000-query cost in USD under different strategy stacks
strategies = [
    "All GPT-4o",
    "All GPT-4o-mini",
    "+ Model Router",
    "+ Semantic Cache (95%)",
    "+ Prompt Caching",
    "+ Batch API (offline)",
]
# cost per 1000 queries (USD), relative to "All GPT-4o" baseline = $25
costs    = [25.0, 1.5, 3.8, 1.9, 1.5, 0.75]
accuracy = [0.98, 0.81, 0.95, 0.95, 0.95, 0.95]  # simulated quality score

fig, ax1 = plt.subplots(figsize=(9, 4))
x = np.arange(len(strategies))
bars = ax1.bar(x, costs, color="#4c72b0", alpha=0.8, width=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(strategies, rotation=18, ha="right", fontsize=9)
ax1.set_ylabel("Cost / 1k queries (USD)", fontsize=10)
ax1.set_ylim(0, 28)
ax1.grid(axis="y", alpha=0.4, linestyle="dashed")

ax2 = ax1.twinx()
ax2.plot(x, accuracy, color="#dd8452", marker="o", linewidth=2, label="Quality score")
ax2.set_ylim(0.75, 1.02)
ax2.set_ylabel("Simulated quality score", fontsize=10)
ax2.legend(loc="upper right", fontsize=9)

for bar, cost in zip(bars, costs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f"${cost}", ha="center", va="bottom", fontsize=8)

ax1.set_title("Cost-quality Pareto curve across optimisation strategies", fontsize=11)
plt.tight_layout(); plt.show()

**Figure.** Cost per 1,000 queries (bars, left axis) and simulated quality score (line, right axis) for each optimisation layer. The model router restores most of the quality lost by downgrading all traffic to GPT-4o-mini. Adding semantic caching and prompt caching cuts cost a further 60% with no quality impact. Batching offline workloads halves the cost of those requests again.

:::{.callout-important}
Never use semantic caching for queries whose answers change over time (e.g. "What is today's SOFR rate?"). Cache entries must be tagged with a validity window and expired when the underlying data source updates.

:::

## Exercises

1. **Benchmark the semantic cache threshold.** Generate 50 paraphrase pairs for five financial questions (10 pairs each) using GPT-4o-mini. For each pair, compute cosine similarity between `text-embedding-3-small` embeddings. Plot a histogram of similarities and choose a threshold that achieves at most 5% false-positive rate (non-duplicate pairs exceeding the threshold).

2. **Implement cache expiry.** Extend `SemanticCache` with a `ttl_seconds: int` parameter. Each `CacheEntry` should store a `created_at` timestamp. The `get` method should skip entries older than `ttl_seconds`. Write a test that seeds the cache, waits `ttl_seconds + 1`, and confirms the entry is no longer returned.

3. **Train a lightweight router classifier.** Collect 100 labelled examples of financial queries annotated with `FAST` or `STRONG`. Train a `sklearn.linear_model.LogisticRegression` on TF-IDF features. Compare its routing accuracy to the rule-based `ModelRouter` on a 20-example holdout set.

---

$\blacksquare$